# 03 - From-scratch DPO on UltraFeedback (helpfulness) (Task 4)

Attach the saved output of notebook 01 as an input (for `outputs/sft/best`). Accelerator: **GPU T4 x2** (policy on cuda:0, frozen reference on cuda:1). ~2-2.5h.

In [ ]:
REPO_URL = "https://github.com/IsrarAliKhan/small-model-alignment-starter.git"  # edit if your fork lives elsewhere
%cd /kaggle/working
!git clone -q {REPO_URL} || git -C small-model-alignment-starter pull -q
%cd /kaggle/working/small-model-alignment-starter
!git log --oneline -1

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors matplotlib pytest
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("CUDA devices:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
import glob

def find(pattern):
    """Locate a file/dir inside attached Kaggle inputs (outputs of earlier notebooks)."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"Not found: {pattern}. Attach the saved output of the notebook that produced it."
    print(pattern, "->", hits[0])
    return hits[0]

In [ ]:
!python scripts/prepare_data.py --task uf

In [ ]:
SFT_BEST = find("outputs/sft/best")
!python scripts/train_dpo.py --sft_checkpoint {SFT_BEST} --train_file data/processed/uf_train.jsonl --val_file data/processed/uf_val.jsonl --output_dir outputs/dpo_uf

In [ ]:
!python scripts/plot_metrics.py --runs outputs/dpo_uf --keys loss reward_accuracy reward_margin --output outputs/plots/dpo_uf.png
from IPython.display import Image; Image('outputs/plots/dpo_uf.png')

In [ ]:
# Small zip of everything except checkpoints (metrics, configs, plots, generations).
!cd outputs && zip -qr ../dpo_uf_run_outputs.zip . -x "*/best/*" "*/final/*" "*/adapter/*" "*/trainer/*" && ls -la ../dpo_uf_run_outputs.zip
print("Now click 'Save Version' (Save & Run All or Quick Save) so outputs persist and can be attached later.")